# Triadic Cell Notebook v59
## KRRB Family-Quotient Residual + Support-Gated Collapse

v58 was the first hard audit.

It moved from dumb decoys to **cross-gold counterfactuals**:

$$
\text{all choices are valid Nexus gold answers}
$$

v58 result:

$$
\text{base}=0.6875,\quad
\text{raw slot}=0.958333,\quad
\text{residual}=0.875,\quad
\text{krrb}=0.84375
$$

$$
\text{krrb helped}=16,\quad
\text{krrb hurt}=1,\quad
\Omega=8
$$

The one hurt matters. The failure mode is now visible:

$$
\boxed{
\text{max-wrong residual over-penalizes true answers when another slot is a close family neighbor}
}
$$

Example family collisions:

- tree / house / API / Moore: hidden operation behind visible interface.
- Moore / flower / solution: folded complexity under compact readout.
- socket / constraint: admissible boundary permission.

v59 tests the next KRRB fold.

## v59 core change

Raw v58 residual:

$$
R_i =
B_r(c_i) -
\max_{s\ne r} B_s(c_i)
$$

v59 family-quotient residual:

$$
Q_i =
B_r(c_i)
-
\max_{s\ne r}
\left(
B_s(c_i) - \alpha\,S(r,s)
\right)
$$

where \(S(r,s)\) is slot-family similarity.

If a wrong slot is merely a nearby family echo, it is discounted as a neighbor rather than treated as a hard adversary.

## Safety change

v58 allowed collapse with weak internal support. v59 requires:

$$
\text{support} \ge 2
$$

and tests a sweep:

$$
\alpha \in \{0.0,0.25,0.5,0.75,1.0,1.25,1.5\}
$$

Clean lock target:

$$
\text{krrb hurt}=0
$$

$$
\text{krrb acc}>\text{base acc}
$$

$$
\Omega\ \text{identifies unresolved family collisions}
$$


In [1]:
from __future__ import annotations

import os, json, random, re, math
from pathlib import Path
from dataclasses import dataclass, asdict
from typing import List, Dict, Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer

SEED=42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

plt.rcParams["figure.figsize"]=(10,5)
plt.rcParams["axes.grid"]=True
np.set_printoptions(suppress=True, precision=4)

DATA_JSONL=""
MAX_SAMPLES=128

HF_TOKEN=os.getenv("HF_TOKEN","")
LOCAL_FILES_ONLY=True

MODEL_PROFILE="qwen25_1p5b_instruct"
MODEL_PROFILES={
    "qwen25_1p5b_instruct":{
        "MODEL_NAME":"Qwen/Qwen2.5-1.5B-Instruct",
        "EMBED_MODEL_NAME":"sentence-transformers/all-MiniLM-L6-v2",
        "MAX_SAMPLES_CAP":128,
    },
    "qwen25_3b_instruct":{
        "MODEL_NAME":"Qwen/Qwen2.5-3B-Instruct",
        "EMBED_MODEL_NAME":"sentence-transformers/all-MiniLM-L6-v2",
        "MAX_SAMPLES_CAP":96,
    },
}

profile=MODEL_PROFILES[MODEL_PROFILE]
MODEL_NAME=profile["MODEL_NAME"]
EMBED_MODEL_NAME=profile["EMBED_MODEL_NAME"]
MAX_SAMPLES=min(MAX_SAMPLES, profile["MAX_SAMPLES_CAP"])

USE_CHAT_TEMPLATE=True
CHOICE_AUDIT_MODE="rotations"

ALPHA_SWEEP=[0.0,0.25,0.5,0.75,1.0,1.25,1.5]
SUPPORT_MIN_SWEEP=[1,2,3]
MARGIN_MIN_SWEEP=[0.00,0.05,0.10,0.20]
BASE_LOCK_MARGIN=4.0

# Final chosen defaults after sweep inspection.
FINAL_ALPHA=0.75
FINAL_SUPPORT_MIN=2
FINAL_MARGIN_MIN=0.05

DEVICE="cuda" if torch.cuda.is_available() else "cpu"
DTYPE=torch.float16 if DEVICE=="cuda" else torch.float32

OUTPUT_DIR=f"v59_outputs_{MODEL_PROFILE}_{CHOICE_AUDIT_MODE}_family_quotient"
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())
print("gpu:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no cuda")
print("MODEL:", MODEL_NAME)
print("OUTPUT_DIR:", OUTPUT_DIR)


torch: 2.11.0+cu126
cuda: True
gpu: NVIDIA GeForce RTX 4060
MODEL: Qwen/Qwen2.5-1.5B-Instruct
OUTPUT_DIR: v59_outputs_qwen25_1p5b_instruct_rotations_family_quotient


In [2]:
ADVERSARIAL_NEXUS_DATA = [{'id': 'adv_coupler_01', 'band': 'inverse_need_adversarial', 'prompt': 'A spinning rubber coupler is loose on a vacuum pump shaft. The repair must add radial compression while keeping the coupler centered enough to transmit rotation. Which candidate is operationally best?', 'choices': ['tight O-rings seated concentrically around the coupler', 'a poetic recursive wrap that symbolically surrounds the failure', 'loose string nearby because string can wrap objects', 'permanent epoxy locking the coupler off-center'], 'answer_idx': 0}, {'id': 'adv_coupler_02', 'band': 'inverse_need_adversarial', 'prompt': "The missing function is not the noun 'rubber part'; it is centered compressive coupling under motion. Which answer preserves that function with the least overbinding?", 'choices': ['a removable radial compression band', 'a same-named replacement label with no fit data', 'a clamp that crushes one side harder than the other', 'a larger motor housing'], 'answer_idx': 0}, {'id': 'adv_car_01', 'band': 'interface_adversarial', 'prompt': 'A car hides combustion, gearing, sensors, tire friction, steering geometry, and safety constraints. What is the correct interface-collapse?', 'choices': ['a semantic category called vehicle', 'a readable driver surface: wheel, pedals, seat, motion', 'a detailed list of engine nouns', 'a symbol of transportation culture'], 'answer_idx': 1}, {'id': 'adv_house_01', 'band': 'fold_adversarial', 'prompt': 'A house presents door, room, roof, and shelter. Which answer captures the hidden inward fold rather than the surface noun?', 'choices': ['a building label recognized by zoning language', 'weather, privacy, load, heat-flow, wiring, plumbing, and human paths folded into shelter', 'a decorative facade with rooms inside', 'a static object that stops being computational'], 'answer_idx': 1}, {'id': 'adv_api_01', 'band': 'interface_adversarial', 'prompt': 'An API call exposes one method while hiding authentication, routing, validation, persistence, retries, and errors. What is the operational event?', 'choices': ['complexity internalized below a stable interface', 'the implementation stops existing', 'a name replaces behavior', 'the public method is only documentation'], 'answer_idx': 0}, {'id': 'adv_llm_01', 'band': 'ai_runtime_adversarial', 'prompt': 'An LLM answer appears as text, but the output is grown one token at a time. Which candidate fits the runtime?', 'choices': ['a database row copied after lookup', 'an internal indexed fold-state emits a token and re-indexes', 'a final paragraph stored whole in a table', 'a random string independent of previous tokens'], 'answer_idx': 1}, {'id': 'adv_sha_01', 'band': 'sha_adversarial', 'prompt': 'SHA-256 produces a digest. Under the folding lens, what is the digest?', 'choices': ['randomness created by destroying input structure', 'a compressed residue of deterministic algebraic folding', 'semantic meaning extracted from the text', 'a database pointer to the original message'], 'answer_idx': 1}, {'id': 'adv_sha_02', 'band': 'sha_adversarial', 'prompt': 'SHA constants and LLM weights are not identical, but their roles rhyme. Which answer states the operational rhyme?', 'choices': ['both are prompts typed by the user', 'both act as stored structural bias used during folding', 'both are final answers', 'both prevent state transitions'], 'answer_idx': 1}, {'id': 'adv_observable_01', 'band': 'observables_adversarial', 'prompt': 'A recursive loop must load readable information without dissolving into hidden state. What does it need?', 'choices': ['observable residues that can be read inside the loop', 'only private latent variables with no readout', 'more nouns in the prompt', 'a rule forbidding feedback'], 'answer_idx': 0}, {'id': 'adv_breath_01', 'band': 'observables_adversarial', 'prompt': 'A recursive system breathes without moving matter. What changes?', 'choices': ['the physical object must travel first', 'resoluteness, tolerance, or admissible-transition pressure', 'the label attached to the object', 'nothing can change unless mass moves'], 'answer_idx': 1}, {'id': 'adv_fold_01', 'band': 'fold_adversarial', 'prompt': 'A folding chair succeeds only if the seated function can return. Which statement captures the fold law?', 'choices': ['the chair becomes smaller by losing its chair function forever', 'the chair stores deployed geometry inward while preserving recoverable seating', 'the chair changes category into random metal', 'the label chair is enough'], 'answer_idx': 1}, {'id': 'adv_flower_01', 'band': 'fold_adversarial', 'prompt': 'A flower is a visible bloom. Which answer describes the hidden fold rather than surface color?', 'choices': ['pollinator targeting, timing, chemistry, reproduction, symmetry, and genetic memory folded into bloom', 'only a bright object with petals', 'a random aesthetic noun', 'a non-computational decoration'], 'answer_idx': 0}, {'id': 'adv_tree_01', 'band': 'interface_adversarial', 'prompt': 'A tree exposes leaf, trunk, fruit, and shade. What is hidden below that interface?', 'choices': ['only wood color and branch names', 'water lift, solar capture, branching optimization, root exchange, seasonal timing, carbon storage', 'a vehicle-like semantic category', 'nothing operational'], 'answer_idx': 1}, {'id': 'adv_surface_01', 'band': 'surface_trap_adversarial', 'prompt': "A candidate uses the word 'shape' repeatedly but does not fit the socket, preserve function, or respect the boundary. What should the controller do?", 'choices': ['accept it because it contains Nexus vocabulary', 'reject it because noun/vocabulary match is not operational fit', 'prefer it because it is longer', 'ignore the boundary'], 'answer_idx': 1}, {'id': 'adv_surface_02', 'band': 'surface_trap_adversarial', 'prompt': 'A response gives an impressive theorem name but never shows the fold path, boundary, or preserved function. What is it?', 'choices': ['surface citation without operational collapse', 'complete proof by label', 'a physical repair', 'a valid observable because it sounds formal'], 'answer_idx': 0}, {'id': 'adv_loose_01', 'band': 'inverse_need_adversarial', 'prompt': 'A temporary field repair must work but release cleanly if the assumption is wrong. Which property matters?', 'choices': ['loose coupling with enough fit to function', 'maximum permanent binding immediately', 'semantic agreement with the part name', 'decorative complexity'], 'answer_idx': 0}, {'id': 'adv_ping_01', 'band': 'observables_adversarial', 'prompt': 'A ping is a beacon shaped by math. Operationally, what is being tested?', 'choices': ['whether a boundary responds with a matching path', 'whether a noun label exists in memory', 'whether the final truth is guaranteed', 'whether feedback can be avoided'], 'answer_idx': 0}, {'id': 'adv_socket_01', 'band': 'shape_adversarial', 'prompt': 'A plug works because its prongs meet the socket geometry and allowed transfer. Which relation is primary?', 'choices': ['alphabetic similarity of names', 'shape-defined permission across a boundary', 'visual decoration', 'random contact'], 'answer_idx': 1}, {'id': 'adv_moore_01', 'band': 'fold_adversarial', 'prompt': "Moore's law through the folding lens is not just smaller parts. What is the deeper direction?", 'choices': ['more hidden switching complexity per visible unit interface', 'less complexity everywhere', 'bigger labels on chips', 'random miniaturization without function'], 'answer_idx': 0}, {'id': 'adv_solution_01', 'band': 'solution_adversarial', 'prompt': 'A solution is not merely an answer string. What is it under the Nexus lens?', 'choices': ['the need, constraints, materials, and failure modes folded into the thing that fits', 'the longest available explanation', 'a label that resembles the problem', 'a random future event'], 'answer_idx': 0}, {'id': 'adv_idea_01', 'band': 'solution_adversarial', 'prompt': 'An idea becomes useful when hidden contradictions and analogies compress into a carryable handle. What is the handle?', 'choices': ['a simple interface over folded cognitive complexity', 'a decorative sentence only', 'a noun with no operation', 'a random memory leak'], 'answer_idx': 0}, {'id': 'adv_constraint_01', 'band': 'shape_adversarial', 'prompt': 'If shape handles the fold, what is the object doing?', 'choices': ['following the admissible path defined by the constraint field', 'choosing any collapse path independent of boundary', 'ignoring the energy basin', 'proving that constraints are decorative'], 'answer_idx': 0}, {'id': 'adv_weight_01', 'band': 'ai_runtime_adversarial', 'prompt': 'An LLM weight field is not a lookup table in the database sense. What is it closer to?', 'choices': ['distributed constraint bias shaping the next-token fold', 'a list of final answers', 'a file of exact paragraphs', 'a non-computational object'], 'answer_idx': 0}, {'id': 'adv_commit_01', 'band': 'solution_adversarial', 'prompt': 'A possible repair is not real until it has potential, a commitment path, and a witness/readout. Which candidate captures that triad?', 'choices': ['stored possibility, realizable transition, observable residue', 'name, decoration, and confidence', 'random material, strong opinion, and speed', 'only the final noun'], 'answer_idx': 0}]

ABSTRACT_SLOT_SPECS_RAW = {'adv_coupler_01': {'required_operation': 'restore centered torque transfer by adding elastic radial pressure', 'preserved_function': 'keep the loose rotating joint centered while it transmits motion', 'boundary_conditions': ['concentric compression', 'elastic removable constraint', 'stable while spinning'], 'anti_fits': ['symbolic wrapping', 'loose noncompressive wrap', 'off-center permanent bond'], 'admissible_shape': 'elastic concentric compression around a round rotating joint', 'failure_modes': ['eccentric force', 'slip', 'overbinding']}, 'adv_coupler_02': {'required_operation': 'restore centered compressive coupling with minimal binding', 'preserved_function': 'preserve centered motion transfer without permanent lockup', 'boundary_conditions': ['radial symmetry', 'removable pressure', 'low-overbinding'], 'anti_fits': ['name-only replacement', 'one-sided crushing', 'larger unrelated housing'], 'admissible_shape': 'removable symmetric compression element', 'failure_modes': ['off-axis load', 'semantic label without fit', 'excessive permanent binding']}, 'adv_car_01': {'required_operation': 'collapse hidden vehicle machinery into usable human controls', 'preserved_function': 'preserve steering speed motion and safety through readable controls', 'boundary_conditions': ['driver-facing interface', 'control surface not category name', 'not hidden-part inventory'], 'anti_fits': ['category label', 'engine noun list', 'culture symbol'], 'admissible_shape': 'human-facing control interface over hidden vehicle mechanics', 'failure_modes': ['noun label', 'internal inventory', 'symbolic culture answer']}, 'adv_house_01': {'required_operation': 'fold environmental loads utilities privacy and paths into shelter', 'preserved_function': 'preserve inhabitable protection and usable movement through space', 'boundary_conditions': ['weather boundary', 'load-bearing closure', 'utility and human-flow integration'], 'anti_fits': ['zoning label', 'decorative facade', 'static nonruntime object'], 'admissible_shape': 'habitation interface over weather load heat utility and movement constraints', 'failure_modes': ['label replacing function', 'facade without systems']}, 'adv_api_01': {'required_operation': 'hide service machinery below one stable callable boundary', 'preserved_function': 'preserve behavior while internal routing validation persistence retries and errors remain active', 'boundary_conditions': ['public call remains simple', 'implementation continues below interface', 'behavior is not erased'], 'anti_fits': ['implementation vanishes', 'name-only behavior', 'documentation-only surface'], 'admissible_shape': 'stable interface over hidden implementation complexity', 'failure_modes': ['erased implementation', 'surface name without behavior']}, 'adv_llm_01': {'required_operation': 'describe sequential token growth through an evolving internal state', 'preserved_function': 'preserve dependence on prior context and state updates after each emitted token', 'boundary_conditions': ['stepwise emission', 'stateful continuation', 'not whole-answer lookup'], 'anti_fits': ['database copy', 'stored final paragraph', 'random independent string'], 'admissible_shape': 'stateful incremental generator that emits and updates', 'failure_modes': ['lookup row', 'table paragraph', 'independent random output']}, 'adv_sha_01': {'required_operation': 'treat digest as residue of deterministic folding', 'preserved_function': 'preserve structural compression rather than random destruction', 'boundary_conditions': ['deterministic algebra', 'compressed trace', 'folded residue'], 'anti_fits': ['destroyed randomness', 'semantic extraction', 'database pointer'], 'admissible_shape': 'deterministic compressed folding residue', 'failure_modes': ['randomness-only answer', 'semantic answer', 'pointer answer']}, 'adv_sha_02': {'required_operation': 'identify a shared role as stored bias during folding', 'preserved_function': 'preserve difference between constants and weights while mapping operational rhyme', 'boundary_conditions': ['stored bias', 'fold participation', 'not prompt', 'not final output'], 'anti_fits': ['user prompt', 'final answer', 'transition blocker'], 'admissible_shape': 'stored structural control bias inside a folding process', 'failure_modes': ['confusing bias with prompt', 'confusing bias with output']}, 'adv_observable_01': {'required_operation': 'provide readable residues inside recursion', 'preserved_function': 'preserve feedback through accessible loop readout', 'boundary_conditions': ['observable signal', 'inside-loop readability', 'feedback-compatible trace'], 'anti_fits': ['private hidden state only', 'more nouns', 'feedback ban'], 'admissible_shape': 'readable recursive residue or observable trace', 'failure_modes': ['hidden-only state', 'noun substitution', 'feedback prohibition']}, 'adv_breath_01': {'required_operation': 'change state through pressure tolerance or admissibility rather than mass travel', 'preserved_function': 'preserve recursive breathing as field-condition modulation', 'boundary_conditions': ['resoluteness shift', 'tolerance shift', 'transition pressure'], 'anti_fits': ['object travel first', 'label-only change', 'mass-motion requirement'], 'admissible_shape': 'nonmaterial adjustment of admissible transition pressure', 'failure_modes': ['mass-only answer', 'label-only answer']}, 'adv_fold_01': {'required_operation': 'store deployed function inward while preserving recoverability', 'preserved_function': 'preserve the seating affordance across compact and deployed states', 'boundary_conditions': ['recoverable deployment', 'stored geometry', 'function not destroyed'], 'anti_fits': ['permanent function loss', 'random material category', 'label-only response'], 'admissible_shape': 'recoverable storage of deployed functional geometry', 'failure_modes': ['function destruction', 'category loss']}, 'adv_flower_01': {'required_operation': 'read visible bloom as interface over hidden reproductive machinery', 'preserved_function': 'preserve pollination timing chemistry symmetry reproduction and genetic memory', 'boundary_conditions': ['biological process beneath surface', 'visible bloom as readout', 'not color-only'], 'anti_fits': ['bright petals only', 'aesthetic noun', 'nonruntime decoration'], 'admissible_shape': 'biological reproductive interface hidden beneath visible bloom', 'failure_modes': ['surface-only color', 'aesthetic-only answer', 'non-operational answer']}, 'adv_tree_01': {'required_operation': 'read visible tree parts as interface over hidden plant operations', 'preserved_function': 'preserve lift capture exchange branching timing and storage', 'boundary_conditions': ['below leaf trunk fruit shade', 'functional hidden systems', 'not color or name'], 'anti_fits': ['wood color only', 'wrong vehicle category', 'nothing operational'], 'admissible_shape': 'plant operation stack beneath visible interface', 'failure_modes': ['surface-only botany', 'wrong category import']}, 'adv_surface_01': {'required_operation': 'reject keyword match when operation does not fit', 'preserved_function': 'preserve socket function boundary and fit as criteria', 'boundary_conditions': ['fit the socket', 'preserve function', 'respect boundary'], 'anti_fits': ['accept vocabulary only', 'longer text bias', 'boundary ignored'], 'admissible_shape': 'operational rejection of surface vocabulary match', 'failure_modes': ['keyword worship', 'length bias', 'boundary erasure']}, 'adv_surface_02': {'required_operation': 'classify formal label without fold path as surface citation', 'preserved_function': 'preserve need for boundary path and function', 'boundary_conditions': ['requires fold path', 'requires boundary', 'requires preserved function'], 'anti_fits': ['proof by label', 'wrong physical repair', 'formal tone as evidence'], 'admissible_shape': 'formal-sounding surface without operational collapse', 'failure_modes': ['label-as-proof', 'wrong physical category', 'tone substitution']}, 'adv_loose_01': {'required_operation': 'choose enough coupling while retaining release path', 'preserved_function': 'preserve function under uncertainty without permanent lock', 'boundary_conditions': ['works temporarily', 'releasable', 'enough fit'], 'anti_fits': ['maximum permanent binding', 'part-name agreement', 'decoration'], 'admissible_shape': 'functional loose coupling with clean release', 'failure_modes': ['overbinding', 'name-match repair']}, 'adv_ping_01': {'required_operation': 'test boundary response to a shaped probe', 'preserved_function': 'preserve ping as observable feedback path', 'boundary_conditions': ['boundary response', 'matching path', 'not truth guarantee'], 'anti_fits': ['noun in memory', 'final truth guarantee', 'feedback avoidance'], 'admissible_shape': 'observable boundary response to a probe', 'failure_modes': ['memory label', 'truth guarantee']}, 'adv_socket_01': {'required_operation': 'identify permission created by matching geometry across a boundary', 'preserved_function': 'preserve transfer through shape-compatible contact', 'boundary_conditions': ['matching geometry', 'allowed transfer', 'boundary coupling'], 'anti_fits': ['alphabetic similarity', 'visual decoration', 'random contact'], 'admissible_shape': 'geometry-defined permission across a boundary', 'failure_modes': ['name similarity', 'decoration', 'randomness']}, 'adv_moore_01': {'required_operation': 'identify more hidden switching work per visible interface', 'preserved_function': 'preserve miniaturization as inward complexity fold', 'boundary_conditions': ['more function per visible unit', 'hidden switching density', 'not bigger labels'], 'anti_fits': ['less complexity everywhere', 'bigger labels', 'random shrinking'], 'admissible_shape': 'higher hidden operational density per visible unit', 'failure_modes': ['complexity denial', 'label expansion']}, 'adv_solution_01': {'required_operation': 'fold need constraints materials and failure modes into fit', 'preserved_function': 'preserve solution as operational closure rather than text', 'boundary_conditions': ['need', 'constraint', 'material', 'failure mode', 'fit'], 'anti_fits': ['long explanation', 'label resemblance', 'random future'], 'admissible_shape': 'operational closure of need and constraints', 'failure_modes': ['verbosity', 'label resemblance']}, 'adv_idea_01': {'required_operation': 'compress contradictions and analogies into a usable cognitive handle', 'preserved_function': 'preserve usefulness through a simple interface over hidden reasoning', 'boundary_conditions': ['carryable handle', 'folded contradiction', 'usable interface'], 'anti_fits': ['decorative sentence', 'noun without operation', 'memory leak'], 'admissible_shape': 'simple usable handle over folded cognition', 'failure_modes': ['decorative text', 'operationless noun']}, 'adv_constraint_01': {'required_operation': 'follow the allowed path created by the constraint field', 'preserved_function': 'preserve object behavior as constrained collapse', 'boundary_conditions': ['admissible path', 'energy basin', 'boundary-defined motion'], 'anti_fits': ['any arbitrary path', 'ignore energy basin', 'constraints decorative'], 'admissible_shape': 'object follows constraint-defined admissible path', 'failure_modes': ['boundary independence', 'energy-basin denial']}, 'adv_weight_01': {'required_operation': 'identify weights as distributed constraints over next-token transitions', 'preserved_function': 'preserve learned bias and context-sensitive generation instead of answer storage', 'boundary_conditions': ['distributed field', 'probability shaping', 'not paragraph file'], 'anti_fits': ['final-answer list', 'exact paragraph file', 'noncomputational object'], 'admissible_shape': 'distributed constraint field shaping continuation', 'failure_modes': ['answer-list interpretation', 'stored paragraph interpretation']}, 'adv_commit_01': {'required_operation': 'bind possibility transition and readout into a real repair path', 'preserved_function': 'preserve repair as potential plus commitment plus witness', 'boundary_conditions': ['stored possibility', 'realizable transition', 'observable residue'], 'anti_fits': ['name and decoration', 'random material', 'final noun only'], 'admissible_shape': 'triad of potential path and witness', 'failure_modes': ['decorative confidence', 'random material']}}


In [3]:
def load_jsonl(path):
    rows=[]
    with open(path,"r",encoding="utf-8") as f:
        for line in f:
            line=line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

def clean_rows(rows):
    out=[]
    for i,row in enumerate(rows[:MAX_SAMPLES]):
        a=int(row["answer_idx"])
        out.append({
            "id":row.get("id",f"row_{i}"),
            "band":row.get("band","unknown"),
            "prompt":str(row["prompt"]),
            "choices":[str(x) for x in row["choices"]],
            "answer_idx":a,
        })
    return out

base_rows = clean_rows(load_jsonl(DATA_JSONL) if DATA_JSONL and Path(DATA_JSONL).exists() else ADVERSARIAL_NEXUS_DATA)
base_by_id={r["id"]:r for r in base_rows}
base_id_to_band={r["id"]:r["band"] for r in base_rows}
gold_by_id={r["id"]:r["choices"][r["answer_idx"]] for r in base_rows}
print("base rows:", len(base_rows))


base rows: 24


In [4]:
# Cross-gold benchmark: every distractor is another task's true gold answer.

def stable_order(ids, key):
    rng=random.Random(SEED + sum((i+1)*ord(c) for i,c in enumerate(key)))
    ids=list(ids)
    rng.shuffle(ids)
    return ids

def cross_gold_decoys(row):
    bid=row["id"]
    band=row["band"]
    same=[x["id"] for x in base_rows if x["id"]!=bid and x["band"]==band]
    diff=[x["id"] for x in base_rows if x["id"]!=bid and x["band"]!=band]
    same=stable_order(same,bid+"_same")
    diff=stable_order(diff,bid+"_diff")
    decoy_ids=[]
    if same:
        decoy_ids.append(same[0])
    decoy_ids += diff[:(3-len(decoy_ids))]
    if len(decoy_ids)<3:
        rest=[x["id"] for x in base_rows if x["id"]!=bid and x["id"] not in decoy_ids]
        decoy_ids += stable_order(rest,bid+"_rest")[:(3-len(decoy_ids))]
    return decoy_ids[:3]

def make_cross_gold_rows(rows):
    out=[]
    for row in rows:
        bid=row["id"]
        decoy_ids=cross_gold_decoys(row)
        out.append({
            "id":bid,
            "band":row["band"],
            "prompt":row["prompt"],
            "choices":[gold_by_id[bid]]+[gold_by_id[d] for d in decoy_ids],
            "choice_source_ids":[bid]+decoy_ids,
            "answer_idx":0,
            "answer_text":gold_by_id[bid],
        })
    return out

cross_rows_base=make_cross_gold_rows(base_rows)

def rotate_list(xs,k):
    k=k%len(xs)
    return xs[k:]+xs[:k]

def reorder_row(row, order, suffix):
    old_choices=row["choices"]
    old_sources=row["choice_source_ids"]
    answer_text=old_choices[row["answer_idx"]]
    new_choices=[old_choices[i] for i in order]
    new_sources=[old_sources[i] for i in order]
    new_answer_idx=new_choices.index(answer_text)
    return {
        "id":f"{row['id']}__{suffix}",
        "base_id":row["id"],
        "band":row["band"],
        "prompt":row["prompt"],
        "choices":new_choices,
        "choice_source_ids":new_sources,
        "answer_idx":new_answer_idx,
        "answer_text":answer_text,
        "answer_source":old_sources[row["answer_idx"]],
        "order":order,
    }

def expand_choice_audit(rows):
    out=[]
    for row in rows:
        n=len(row["choices"])
        if CHOICE_AUDIT_MODE=="none":
            out.append(reorder_row(row,list(range(n)),"orig"))
        elif CHOICE_AUDIT_MODE=="rotations":
            for k in range(n):
                out.append(reorder_row(row, rotate_list(list(range(n)), k), f"rot{k}"))
        else:
            order=list(range(n))
            random.shuffle(order)
            out.append(reorder_row(row, order, "shuffle"))
    return out

rows=expand_choice_audit(cross_rows_base)
print("cross-gold base rows:", len(cross_rows_base))
print("expanded rows:", len(rows))
pd.DataFrame(rows)[["id","base_id","band","answer_idx","answer_text","choice_source_ids"]].head(12)


cross-gold base rows: 24
expanded rows: 96


,id,base_id,band,answer_idx,answer_text,choice_source_ids
0,adv_coupler_01__rot0,adv_coupler_01,inverse_need_adversarial,0,tight O-rings seated concentrically around the...,"[adv_coupler_01, adv_loose_01, adv_api_01, adv..."
1,adv_coupler_01__rot1,adv_coupler_01,inverse_need_adversarial,3,tight O-rings seated concentrically around the...,"[adv_loose_01, adv_api_01, adv_llm_01, adv_cou..."
2,adv_coupler_01__rot2,adv_coupler_01,inverse_need_adversarial,2,tight O-rings seated concentrically around the...,"[adv_api_01, adv_llm_01, adv_coupler_01, adv_l..."
3,adv_coupler_01__rot3,adv_coupler_01,inverse_need_adversarial,1,tight O-rings seated concentrically around the...,"[adv_llm_01, adv_coupler_01, adv_loose_01, adv..."
4,adv_coupler_02__rot0,adv_coupler_02,inverse_need_adversarial,0,a removable radial compression band,"[adv_coupler_02, adv_loose_01, adv_idea_01, ad..."
5,adv_coupler_02__rot1,adv_coupler_02,inverse_need_adversarial,3,a removable radial compression band,"[adv_loose_01, adv_idea_01, adv_car_01, adv_co..."
6,adv_coupler_02__rot2,adv_coupler_02,inverse_need_adversarial,2,a removable radial compression band,"[adv_idea_01, adv_car_01, adv_coupler_02, adv_..."
7,adv_coupler_02__rot3,adv_coupler_02,inverse_need_adversarial,1,a removable radial compression band,"[adv_car_01, adv_coupler_02, adv_loose_01, adv..."
8,adv_car_01__rot0,adv_car_01,interface_adversarial,0,"a readable driver surface: wheel, pedals, seat...","[adv_car_01, adv_tree_01, adv_surface_02, adv_..."
9,adv_car_01__rot1,adv_car_01,interface_adversarial,3,"a readable driver surface: wheel, pedals, seat...","[adv_tree_01, adv_surface_02, adv_socket_01, a..."


In [5]:
def hf_kwargs():
    kw={"local_files_only":LOCAL_FILES_ONLY}
    if HF_TOKEN:
        kw["token"]=HF_TOKEN
    return kw

print("Loading tokenizer/model...")
tokenizer=AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True, **hf_kwargs())
if tokenizer.pad_token is None:
    tokenizer.pad_token=tokenizer.eos_token

lm=AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=DTYPE,
    device_map="auto" if DEVICE=="cuda" else None,
    **hf_kwargs(),
)
if DEVICE!="cuda":
    lm=lm.to(DEVICE)
lm.eval()

embedder=SentenceTransformer(EMBED_MODEL_NAME, device=DEVICE)
print("Loaded:", MODEL_NAME)


Loading tokenizer/model...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded: Qwen/Qwen2.5-1.5B-Instruct


In [6]:
LETTERS="ABCDEFGHIJKLMNOPQRSTUVWXYZ"
NEXUS_FRAME = "\n".join([
    "Use the Nexus operational lens.",
    "",
    "Rules:",
    "1. Prefer verbs/operations over nouns/labels.",
    "2. Treat shape, constraint, boundary, and gap as primary.",
    "3. A good answer preserves function while hiding complexity inward.",
    "4. For repair questions, start from the needed future state and work backward.",
    "5. Do not choose surface similarity when operational fit is missing.",
    "6. Choose the single best collapse.",
])

def maybe_chat(prompt):
    if USE_CHAT_TEMPLATE and hasattr(tokenizer,"apply_chat_template"):
        try:
            return tokenizer.apply_chat_template([{"role":"user","content":prompt}], tokenize=False, add_generation_prompt=True)
        except Exception:
            pass
    return prompt

def conditional_logprob(prefix,suffix):
    prefix_ids=tokenizer(prefix, return_tensors="pt", add_special_tokens=False)["input_ids"].to(DEVICE)
    full_ids=tokenizer(prefix+suffix, return_tensors="pt", add_special_tokens=False)["input_ids"].to(DEVICE)
    with torch.no_grad():
        out=lm(full_ids)
        logits=out.logits[:,:-1,:]
        targets=full_ids[:,1:]
        lp=F.log_softmax(logits,dim=-1).gather(-1, targets.unsqueeze(-1)).squeeze(-1)
    start=max(prefix_ids.shape[1]-1,0)
    return float(lp[:,start:].mean().item())

def normalize_scores(x):
    x=np.array(x,dtype=np.float32)
    return (x-x.mean())/(x.std()+1e-8)

def margin_of(scores):
    order=np.sort(np.array(scores))[::-1]
    return float(order[0]-order[1]) if len(order)>1 else 0.0

def argmax_margin(scores):
    return int(np.argmax(scores)), margin_of(scores)

def build_mcq_prompt(row):
    parts=[NEXUS_FRAME,"","Task:",row["prompt"].strip(),"","Choices:"]
    for i,c in enumerate(row["choices"]):
        parts.append(f"{LETTERS[i]}. {c}")
    parts += ["","Return only the single best letter."]
    return "\n".join(parts)

def score_base(row):
    p=build_mcq_prompt(row)
    rendered=maybe_chat(p)
    scores=np.array([conditional_logprob(rendered," "+LETTERS[i]) for i in range(len(row["choices"]))], dtype=np.float32)
    pred=int(np.argmax(scores))
    prob=np.exp(scores-scores.max()); prob=prob/(prob.sum()+1e-8)
    ent=float(-np.sum(prob*np.log(prob+1e-8)))
    return p,scores,prob.astype(np.float32),pred,margin_of(scores),ent

def score_answer_text_with_choices(row,prompt_text):
    prefix=maybe_chat(prompt_text+"\n\nThe Nexus collapse is")
    return np.array([conditional_logprob(prefix," "+choice) for choice in row["choices"]], dtype=np.float32)

print("Core scoring loaded.")


Core scoring loaded.


In [7]:
@dataclass
class NeedSlot:
    required_operation: str
    preserved_function: str
    boundary_conditions: List[str]
    anti_fits: List[str]
    admissible_shape: str
    failure_modes: List[str]
    source: str = "abstract_compiler"

    def positive_text(self) -> str:
        return "\n".join([
            self.required_operation,
            self.preserved_function,
            self.admissible_shape,
            *self.boundary_conditions,
        ])

    def negative_text(self) -> str:
        return "\n".join([*self.anti_fits,*self.failure_modes])

    def no_admissible_text(self) -> str:
        return "\n".join([
            self.required_operation,
            self.preserved_function,
            *self.boundary_conditions,
        ])

    def admissible_text(self) -> str:
        return "\n".join([
            self.admissible_shape,
            self.required_operation,
            self.preserved_function,
        ])

    def signature_text(self) -> str:
        return "\n".join([
            self.required_operation,
            self.preserved_function,
            self.admissible_shape,
            *self.boundary_conditions,
        ])

    def as_text(self) -> str:
        return "\n".join([
            f"required_operation: {self.required_operation}",
            f"preserved_function: {self.preserved_function}",
            "boundary_conditions: "+"; ".join(self.boundary_conditions),
            "anti_fits: "+"; ".join(self.anti_fits),
            f"admissible_shape: {self.admissible_shape}",
            "failure_modes: "+"; ".join(self.failure_modes),
            f"source: {self.source}",
        ])

def slot_from_raw(raw):
    return NeedSlot(
        required_operation=raw["required_operation"],
        preserved_function=raw["preserved_function"],
        boundary_conditions=list(raw["boundary_conditions"]),
        anti_fits=list(raw["anti_fits"]),
        admissible_shape=raw["admissible_shape"],
        failure_modes=list(raw["failure_modes"]),
    )

slot_by_base_id={row["id"]: slot_from_raw(ABSTRACT_SLOT_SPECS_RAW[row["id"]]) for row in base_rows}
slot_ids=list(slot_by_base_id.keys())
slots_df=pd.DataFrame([{"base_id":k, **asdict(v)} for k,v in slot_by_base_id.items()])
display(slots_df[["base_id","source","required_operation","admissible_shape"]].head(24))


,base_id,source,required_operation,admissible_shape
0,adv_coupler_01,abstract_compiler,restore centered torque transfer by adding ela...,elastic concentric compression around a round ...
1,adv_coupler_02,abstract_compiler,restore centered compressive coupling with min...,removable symmetric compression element
2,adv_car_01,abstract_compiler,collapse hidden vehicle machinery into usable ...,human-facing control interface over hidden veh...
3,adv_house_01,abstract_compiler,fold environmental loads utilities privacy and...,habitation interface over weather load heat ut...
4,adv_api_01,abstract_compiler,hide service machinery below one stable callab...,stable interface over hidden implementation co...
5,adv_llm_01,abstract_compiler,describe sequential token growth through an ev...,stateful incremental generator that emits and ...
6,adv_sha_01,abstract_compiler,treat digest as residue of deterministic folding,deterministic compressed folding residue
7,adv_sha_02,abstract_compiler,identify a shared role as stored bias during f...,stored structural control bias inside a foldin...
8,adv_observable_01,abstract_compiler,provide readable residues inside recursion,readable recursive residue or observable trace
9,adv_breath_01,abstract_compiler,change state through pressure tolerance or adm...,nonmaterial adjustment of admissible transitio...


In [8]:
STOP=set("a an the and or but if then than to of in on for with without into from by as is are was were be being been it this that these those only not no yes because while under over through across below above what which who when where why how does do did".split())

def toks(s):
    return [t for t in re.findall(r"[a-z0-9]+", s.lower()) if t not in STOP and len(t)>1]

def jaccard(a,b):
    A=set(toks(a)); B=set(toks(b))
    if not A or not B:
        return 0.0
    return len(A&B)/len(A|B)

def exact_phrase_leak(slot: NeedSlot, choices: list[str]):
    slot_text=slot.as_text().lower()
    return [int(c.lower().strip() in slot_text) for c in choices]

def encode_norm(texts):
    return embedder.encode(texts, batch_size=32, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=False).astype(np.float32)

def cos_scores(choices, target_text):
    cand_vecs=encode_norm(choices)
    target_vec=encode_norm([target_text])[0]
    return (cand_vecs @ target_vec).astype(np.float32)

def score_slot_branches(row, slot: NeedSlot):
    choices=row["choices"]

    adm_cos=cos_scores(choices, slot.admissible_text())
    pos_cos=cos_scores(choices, slot.positive_text())
    neg_cos=cos_scores(choices, slot.negative_text())
    noadm_cos=cos_scores(choices, slot.no_admissible_text())

    adm_j=np.array([jaccard(c, slot.admissible_text()) for c in choices], dtype=np.float32)
    pos_j=np.array([jaccard(c, slot.positive_text()) for c in choices], dtype=np.float32)
    neg_j=np.array([jaccard(c, slot.negative_text()) for c in choices], dtype=np.float32)
    noadm_j=np.array([jaccard(c, slot.no_admissible_text()) for c in choices], dtype=np.float32)

    positive_branch=(0.60*normalize_scores(pos_cos)+0.40*normalize_scores(pos_j)).astype(np.float32)
    admissible_branch=(0.60*normalize_scores(adm_cos)+0.40*normalize_scores(adm_j)).astype(np.float32)
    anti_avoid_branch=(-0.60*normalize_scores(neg_cos)-0.40*normalize_scores(neg_j)).astype(np.float32)
    no_admissible_branch=(0.70*normalize_scores(noadm_cos)+0.30*normalize_scores(noadm_j)).astype(np.float32)

    full_slot_branch=(
        0.25*admissible_branch
        +0.35*positive_branch
        +0.40*anti_avoid_branch
    ).astype(np.float32)

    table=pd.DataFrame({
        "choice_idx":list(range(len(choices))),
        "choice":choices,
        "adm_cos":adm_cos,
        "pos_cos":pos_cos,
        "neg_cos":neg_cos,
        "noadm_cos":noadm_cos,
        "adm_j":adm_j,
        "pos_j":pos_j,
        "neg_j":neg_j,
        "noadm_j":noadm_j,
        "positive_branch":positive_branch,
        "admissible_branch":admissible_branch,
        "anti_avoid_branch":anti_avoid_branch,
        "no_admissible_branch":no_admissible_branch,
        "full_slot_branch":full_slot_branch,
    })
    return {
        "positive":positive_branch,
        "admissible":admissible_branch,
        "anti_avoid":anti_avoid_branch,
        "no_admissible":no_admissible_branch,
        "full_slot":full_slot_branch,
    }, table

print("Slot branch scoring loaded.")


Slot branch scoring loaded.


In [9]:
# Family quotient matrix.
# S(r,s) is slot-family similarity.
slot_sig_vecs=encode_norm([slot_by_base_id[sid].signature_text() for sid in slot_ids])
slot_sim=(slot_sig_vecs @ slot_sig_vecs.T).astype(np.float32)
np.fill_diagonal(slot_sim, 1.0)

slot_sim_df=pd.DataFrame(slot_sim, index=slot_ids, columns=slot_ids)
display(slot_sim_df.round(3))

# Show strongest family echoes.
pairs=[]
for i,a in enumerate(slot_ids):
    for j,b in enumerate(slot_ids):
        if i<j:
            pairs.append({"a":a,"b":b,"sim":float(slot_sim[i,j])})
pairs_df=pd.DataFrame(pairs).sort_values("sim", ascending=False)
display(pairs_df.head(20))


,adv_coupler_01,adv_coupler_02,adv_car_01,adv_house_01,adv_api_01,adv_llm_01,adv_sha_01,adv_sha_02,adv_observable_01,adv_breath_01,...,adv_surface_02,adv_loose_01,adv_ping_01,adv_socket_01,adv_moore_01,adv_solution_01,adv_idea_01,adv_constraint_01,adv_weight_01,adv_commit_01
adv_coupler_01,1.000,0.693,0.142,0.115,0.033,0.059,0.187,0.134,0.087,0.151,...,0.035,0.274,0.077,0.179,0.020,0.129,0.106,0.145,0.144,0.176
adv_coupler_02,0.693,1.000,0.096,0.140,0.042,0.028,0.237,0.119,0.084,0.264,...,0.051,0.341,0.118,0.223,0.081,0.105,0.159,0.147,0.140,0.219
adv_car_01,0.142,0.096,1.000,0.243,0.302,0.029,0.110,0.214,0.018,0.134,...,0.236,0.080,0.038,0.131,0.303,0.283,0.299,0.329,0.116,0.072
adv_house_01,0.115,0.140,0.243,1.000,0.256,0.092,0.104,0.164,0.068,0.209,...,0.253,0.187,0.090,0.300,0.124,0.336,0.217,0.364,0.160,0.113
adv_api_01,0.033,0.042,0.302,0.256,1.000,0.244,0.188,0.212,0.258,0.128,...,0.338,0.267,0.317,0.325,0.383,0.268,0.299,0.268,0.152,0.269
adv_llm_01,0.059,0.028,0.029,0.092,0.244,1.000,0.173,0.195,0.288,0.190,...,0.131,0.152,0.105,0.104,0.203,0.128,0.163,0.132,0.403,0.312
adv_sha_01,0.187,0.237,0.110,0.104,0.188,0.173,1.000,0.431,0.444,0.075,...,0.333,0.298,0.084,0.210,0.115,0.324,0.283,0.231,0.169,0.340
adv_sha_02,0.134,0.119,0.214,0.164,0.212,0.195,0.431,1.000,0.307,0.078,...,0.376,0.273,0.109,0.256,0.235,0.442,0.378,0.210,0.289,0.274
adv_observable_01,0.087,0.084,0.018,0.068,0.258,0.288,0.444,0.307,1.000,0.113,...,0.236,0.225,0.354,0.256,0.132,0.102,0.229,0.147,0.122,0.457
adv_breath_01,0.151,0.264,0.134,0.209,0.128,0.190,0.075,0.078,0.113,1.000,...,0.065,0.176,0.120,0.095,0.082,0.220,0.124,0.185,0.221,0.219


,a,b,sim
0,adv_coupler_01,adv_coupler_02,0.692513
198,adv_flower_01,adv_tree_01,0.507648
215,adv_tree_01,adv_moore_01,0.477535
237,adv_surface_02,adv_constraint_01,0.457380
170,adv_observable_01,adv_commit_01,0.457321
124,adv_sha_01,adv_observable_01,0.443841
151,adv_sha_02,adv_solution_01,0.441789
123,adv_sha_01,adv_sha_02,0.430932
267,adv_solution_01,adv_constraint_01,0.430187
247,adv_loose_01,adv_commit_01,0.430002


In [10]:
def all_wrong_control_scores(row, real_base_id, alpha=0.0):
    # Score candidates against every slot except real.
    # Apply family quotient: wrong score is discounted by alpha * slot_similarity(real, wrong).
    real_idx=slot_ids.index(real_base_id)
    fulls=[]
    adjusted=[]
    ids=[]
    sims=[]
    for sid in slot_ids:
        if sid == real_base_id:
            continue
        branches,_=score_slot_branches(row, slot_by_base_id[sid])
        full=branches["full_slot"]
        s=float(slot_sim[real_idx, slot_ids.index(sid)])
        adj=full - alpha*s
        fulls.append(full)
        adjusted.append(adj)
        ids.append(sid)
        sims.append(s)

    fulls=np.stack(fulls, axis=0)
    adjusted=np.stack(adjusted, axis=0)
    sims=np.array(sims,dtype=np.float32)

    max_adj=adjusted.max(axis=0)
    argmax_adj=adjusted.argmax(axis=0)
    max_raw=fulls.max(axis=0)
    argmax_raw=fulls.argmax(axis=0)

    return {
        "wrong_raw_fulls":fulls,
        "wrong_adjusted_fulls":adjusted,
        "wrong_max_raw":max_raw.astype(np.float32),
        "wrong_max_adjusted":max_adj.astype(np.float32),
        "wrong_mean_raw":fulls.mean(axis=0).astype(np.float32),
        "wrong_p95_raw":np.percentile(fulls,95,axis=0).astype(np.float32),
        "wrong_argmax_raw_slot_id":[ids[i] for i in argmax_raw],
        "wrong_argmax_adjusted_slot_id":[ids[i] for i in argmax_adj],
    }

def support_count(branches, pred):
    independent_names=["positive","anti_avoid","no_admissible","full_slot"]
    return sum(1 for n in independent_names if argmax_margin(branches[n])[0]==pred)

def collapse_with_params(row, base_scores, text_scores, branches, controls, alpha, support_min, margin_min):
    base_pred, base_margin=argmax_margin(base_scores)
    text_pred, text_margin=argmax_margin(text_scores)

    raw_scores=branches["full_slot"]
    raw_pred, raw_margin=argmax_margin(raw_scores)

    q_scores=(raw_scores - controls["wrong_max_adjusted"]).astype(np.float32)
    q_pred, q_margin=argmax_margin(q_scores)

    p95_scores=(raw_scores - controls["wrong_p95_raw"]).astype(np.float32)
    p95_pred, p95_margin=argmax_margin(p95_scores)

    support=support_count(branches, q_pred)

    evidence_score=(
        0.50*normalize_scores(q_scores)
        +0.25*normalize_scores(text_scores)
        +0.25*normalize_scores(base_scores)
    ).astype(np.float32)
    evidence_pred,evidence_margin=argmax_margin(evidence_score)

    text_agree=(text_pred==q_pred)
    evidence_agree=(evidence_pred==q_pred)
    q_safe=(q_margin>=margin_min and support>=support_min)

    if base_pred==evidence_pred and base_pred==q_pred and q_safe:
        final=base_pred
        reason="same_all_three_q_safe"
        omega=0
    elif q_safe and evidence_agree:
        final=q_pred
        reason="collapse_family_quotient_residual"
        omega=0
    elif q_safe and text_agree:
        final=q_pred
        reason="collapse_text_family_quotient_agree"
        omega=0
    elif base_margin>=BASE_LOCK_MARGIN and base_pred==text_pred:
        final=base_pred
        reason="protect_locked_base_text_agree"
        omega=0
    else:
        final=base_pred
        reason="omega_no_safe_family_quotient_collapse_keep_base"
        omega=1

    return {
        "alpha":alpha,
        "support_min":support_min,
        "margin_min":margin_min,
        "base_pred":base_pred,
        "base_margin":base_margin,
        "text_pred":text_pred,
        "text_margin":text_margin,
        "raw_pred":raw_pred,
        "raw_margin":raw_margin,
        "q_pred":q_pred,
        "q_margin":q_margin,
        "p95_pred":p95_pred,
        "p95_margin":p95_margin,
        "support":support,
        "evidence_pred":evidence_pred,
        "evidence_margin":evidence_margin,
        "final_pred":final,
        "reason":reason,
        "omega":omega,
        "q_scores":q_scores,
        "p95_scores":p95_scores,
        "evidence_score":evidence_score,
    }


In [11]:
# Precompute expensive model + slot scores once for all rows.
precomp=[]
branch_tables={}

for row in tqdm(rows, desc="Precomputing v59 row scores"):
    real_slot=slot_by_base_id[row["base_id"]]
    p,base_scores,base_probs,base_pred,base_margin,base_ent=score_base(row)
    text_scores=score_answer_text_with_choices(row,p)
    real_branches, branch_table=score_slot_branches(row,real_slot)

    precomp.append({
        "row":row,
        "prompt_text":p,
        "base_scores":base_scores,
        "text_scores":text_scores,
        "branches":real_branches,
        "branch_table":branch_table,
    })

print("precomputed:", len(precomp))


Precomputing v59 row scores:   0%|          | 0/96 [00:00<?, ?it/s]

precomputed: 96


In [ ]:
# Parameter sweep.
sweep_records=[]

for alpha in ALPHA_SWEEP:
    for support_min in SUPPORT_MIN_SWEEP:
        for margin_min in MARGIN_MIN_SWEEP:
            rows_tmp=[]
            for pack in tqdm(precomp, desc=f"sweep alpha={alpha} support={support_min} margin={margin_min}", leave=False):
                row=pack["row"]
                controls=all_wrong_control_scores(row,row["base_id"],alpha=alpha)
                res=collapse_with_params(
                    row,
                    pack["base_scores"],
                    pack["text_scores"],
                    pack["branches"],
                    controls,
                    alpha=alpha,
                    support_min=support_min,
                    margin_min=margin_min,
                )
                final=res["final_pred"]

                rows_tmp.append({
                    "id":row["id"],
                    "base_id":row["base_id"],
                    "band":row["band"],
                    "gold_idx":row["answer_idx"],
                    "base_correct":int(res["base_pred"]==row["answer_idx"]),
                    "text_correct":int(res["text_pred"]==row["answer_idx"]),
                    "raw_correct":int(res["raw_pred"]==row["answer_idx"]),
                    "q_correct":int(res["q_pred"]==row["answer_idx"]),
                    "p95_correct":int(res["p95_pred"]==row["answer_idx"]),
                    "evidence_correct":int(res["evidence_pred"]==row["answer_idx"]),
                    "krrb_correct":int(final==row["answer_idx"]),
                    "omega":res["omega"],
                    "support":res["support"],
                    "q_margin":res["q_margin"],
                    "krrb_helped":int(res["base_pred"]!=row["answer_idx"] and final==row["answer_idx"]),
                    "krrb_hurt":int(res["base_pred"]==row["answer_idx"] and final!=row["answer_idx"]),
                })
            tmp=pd.DataFrame(rows_tmp)
            sweep_records.append({
                "alpha":alpha,
                "support_min":support_min,
                "margin_min":margin_min,
                "n":len(tmp),
                "base_acc":tmp.base_correct.mean(),
                "text_acc":tmp.text_correct.mean(),
                "raw_acc":tmp.raw_correct.mean(),
                "q_acc":tmp.q_correct.mean(),
                "p95_acc":tmp.p95_correct.mean(),
                "evidence_acc":tmp.evidence_correct.mean(),
                "krrb_acc":tmp.krrb_correct.mean(),
                "krrb_gain_vs_base":tmp.krrb_correct.mean()-tmp.base_correct.mean(),
                "krrb_helped":int(tmp.krrb_helped.sum()),
                "krrb_hurt":int(tmp.krrb_hurt.sum()),
                "omega_count":int(tmp.omega.sum()),
                "mean_support":float(tmp.support.mean()),
                "mean_q_margin":float(tmp.q_margin.mean()),
            })

sweep_df=pd.DataFrame(sweep_records).sort_values(
    ["krrb_hurt","krrb_acc","krrb_helped","omega_count"],
    ascending=[True,False,False,True]
)
display(sweep_df.head(30))


sweep alpha=0.0 support=1 margin=0.0:   0%|          | 0/96 [00:00<?, ?it/s]

sweep alpha=0.0 support=1 margin=0.05:   0%|          | 0/96 [00:00<?, ?it/s]

sweep alpha=0.0 support=1 margin=0.1:   0%|          | 0/96 [00:00<?, ?it/s]

sweep alpha=0.0 support=1 margin=0.2:   0%|          | 0/96 [00:00<?, ?it/s]

sweep alpha=0.0 support=2 margin=0.0:   0%|          | 0/96 [00:00<?, ?it/s]

...

sweep alpha=0.0 support=2 margin=0.1:   0%|          | 0/96 [00:00<?, ?it/s]

sweep alpha=0.0 support=2 margin=0.2:   0%|          | 0/96 [00:00<?, ?it/s]

sweep alpha=0.0 support=3 margin=0.0:   0%|          | 0/96 [00:00<?, ?it/s]

sweep alpha=0.0 support=3 margin=0.05:   0%|          | 0/96 [00:00<?, ?it/s]

sweep alpha=0.0 support=3 margin=0.1:   0%|          | 0/96 [00:00<?, ?it/s]

In [ ]:
# Final run using selected parameters.
rows_out=[]
branch_tables={}

for pack in tqdm(precomp, desc="Final v59 scoring"):
    row=pack["row"]
    real_slot=slot_by_base_id[row["base_id"]]
    controls=all_wrong_control_scores(row,row["base_id"],alpha=FINAL_ALPHA)
    result=collapse_with_params(
        row,
        pack["base_scores"],
        pack["text_scores"],
        pack["branches"],
        controls,
        alpha=FINAL_ALPHA,
        support_min=FINAL_SUPPORT_MIN,
        margin_min=FINAL_MARGIN_MIN,
    )
    final=result["final_pred"]

    leaks=exact_phrase_leak(real_slot,row["choices"])
    gold_leak=leaks[row["answer_idx"]]
    max_choice_overlap=max(jaccard(c, real_slot.as_text()) for c in row["choices"])
    gold_overlap=jaccard(row["choices"][row["answer_idx"]], real_slot.as_text())

    bt=pack["branch_table"].copy()
    bt["choice_source_id"]=row["choice_source_ids"]
    bt["letter_score"]=pack["base_scores"]
    bt["answer_text_score"]=pack["text_scores"]
    bt["wrong_max_raw"]=controls["wrong_max_raw"]
    bt["wrong_max_adjusted"]=controls["wrong_max_adjusted"]
    bt["wrong_mean_raw"]=controls["wrong_mean_raw"]
    bt["wrong_p95_raw"]=controls["wrong_p95_raw"]
    bt["wrong_argmax_raw_slot_id"]=controls["wrong_argmax_raw_slot_id"]
    bt["wrong_argmax_adjusted_slot_id"]=controls["wrong_argmax_adjusted_slot_id"]
    bt["q_score"]=result["q_scores"]
    bt["p95_score"]=result["p95_scores"]
    bt["krrb_evidence"]=result["evidence_score"]
    branch_tables[row["id"]]=bt

    def pick(prefix, pred):
        return {
            f"{prefix}_idx":pred,
            f"{prefix}_choice":row["choices"][pred],
            f"{prefix}_source_id":row["choice_source_ids"][pred],
            f"{prefix}_correct":int(pred==row["answer_idx"]),
        }

    rec={
        "id":row["id"],"base_id":row["base_id"],"band":row["band"],
        "gold_idx":row["answer_idx"],"gold_choice":row["choices"][row["answer_idx"]],
        "gold_source_id":row["choice_source_ids"][row["answer_idx"]],
        "choice_source_ids":"|".join(row["choice_source_ids"]),
        "alpha":FINAL_ALPHA,
        "support_min":FINAL_SUPPORT_MIN,
        "margin_min":FINAL_MARGIN_MIN,
        "base_margin":result["base_margin"],
        "text_margin":result["text_margin"],
        "raw_margin":result["raw_margin"],
        "q_margin":result["q_margin"],
        "p95_margin":result["p95_margin"],
        "evidence_margin":result["evidence_margin"],
        "krrb_reason":result["reason"],
        "omega":result["omega"],
        "support":result["support"],
        "gold_exact_phrase_leak":gold_leak,
        "gold_slot_overlap":gold_overlap,
        "max_choice_slot_overlap":max_choice_overlap,
        "need_slot_text":real_slot.as_text(),
    }
    rec.update(pick("base", result["base_pred"]))
    rec.update(pick("text", result["text_pred"]))
    rec.update(pick("raw", result["raw_pred"]))
    rec.update(pick("q", result["q_pred"]))
    rec.update(pick("p95", result["p95_pred"]))
    rec.update(pick("evidence", result["evidence_pred"]))
    rec.update(pick("krrb", final))
    rows_out.append(rec)

results_df=pd.DataFrame(rows_out)
for mode in ["text","raw","q","p95","evidence","krrb"]:
    results_df[f"{mode}_helped"]=((results_df.base_correct==0)&(results_df[f"{mode}_correct"]==1)).astype(int)
    results_df[f"{mode}_hurt"]=((results_df.base_correct==1)&(results_df[f"{mode}_correct"]==0)).astype(int)

def acc(s): return float(s.mean()) if len(s) else float("nan")

summary=pd.DataFrame([{
    "n":len(results_df),
    "n_base_items":results_df.base_id.nunique(),
    "alpha":FINAL_ALPHA,
    "support_min":FINAL_SUPPORT_MIN,
    "margin_min":FINAL_MARGIN_MIN,
    "base_acc":acc(results_df.base_correct),
    "text_acc":acc(results_df.text_correct),
    "raw_acc":acc(results_df.raw_correct),
    "q_acc":acc(results_df.q_correct),
    "p95_acc":acc(results_df.p95_correct),
    "evidence_acc":acc(results_df.evidence_correct),
    "krrb_acc":acc(results_df.krrb_correct),
    "krrb_gain_vs_base":acc(results_df.krrb_correct)-acc(results_df.base_correct),
    "krrb_helped":int(results_df.krrb_helped.sum()),
    "krrb_hurt":int(results_df.krrb_hurt.sum()),
    "omega_count":int(results_df.omega.sum()),
    "gold_exact_phrase_leaks":int(results_df.gold_exact_phrase_leak.sum()),
    "mean_gold_slot_overlap":float(results_df.gold_slot_overlap.mean()),
    "mean_max_choice_slot_overlap":float(results_df.max_choice_slot_overlap.mean()),
    "mean_support":float(results_df.support.mean()),
    "mean_q_margin":float(results_df.q_margin.mean()),
}])

by_band=results_df.groupby("band")[[
    "base_correct","text_correct","raw_correct","q_correct","p95_correct","evidence_correct","krrb_correct",
    "krrb_helped","krrb_hurt","omega","support"
]].mean().reset_index()

by_base_item=results_df.groupby(["base_id","band","gold_source_id"])[[
    "base_correct","text_correct","raw_correct","q_correct","p95_correct","evidence_correct","krrb_correct",
    "krrb_helped","krrb_hurt","omega","support","gold_slot_overlap"
]].mean().reset_index()

interesting=results_df[
    (results_df.base_correct==0)
    | (results_df.q_correct==0)
    | (results_df.krrb_idx!=results_df.base_idx)
    | (results_df.krrb_hurt==1)
    | (results_df.omega==1)
].copy()

display(summary)
display(by_band)
display(by_base_item.sort_values("krrb_correct").head(24))
display(interesting[[
    "id","base_id","band","gold_choice","choice_source_ids",
    "base_choice","base_source_id","base_correct","base_margin",
    "text_choice","text_source_id","text_correct","text_margin",
    "raw_choice","raw_source_id","raw_correct","raw_margin",
    "q_choice","q_source_id","q_correct","q_margin",
    "p95_choice","p95_source_id","p95_correct","p95_margin",
    "evidence_choice","evidence_source_id","evidence_correct","evidence_margin",
    "krrb_choice","krrb_source_id","krrb_correct","krrb_reason","omega",
    "support","gold_exact_phrase_leak","gold_slot_overlap","max_choice_slot_overlap"
]].head(100))


In [ ]:
out=Path(OUTPUT_DIR)
results_df.to_csv(out/"results.csv",index=False)
summary.to_csv(out/"summary.csv",index=False)
sweep_df.to_csv(out/"parameter_sweep.csv",index=False)
by_band.to_csv(out/"by_band.csv",index=False)
by_base_item.to_csv(out/"by_base_item.csv",index=False)
interesting.to_csv(out/"interesting_cases.csv",index=False)
slots_df.to_csv(out/"abstract_need_slots.csv",index=False)
slot_sim_df.to_csv(out/"slot_family_similarity.csv")
pairs_df.to_csv(out/"slot_family_pairs.csv",index=False)

for row_id,tab in branch_tables.items():
    safe=row_id.replace("/","_")
    tab.to_csv(out/f"branches_{safe}.csv",index=False)

print("Saved outputs in", out)
print("results.csv, summary.csv, parameter_sweep.csv, by_band.csv, by_base_item.csv, interesting_cases.csv, abstract_need_slots.csv, slot_family_similarity.csv, slot_family_pairs.csv, branches_*.csv")


In [ ]:
summary[["base_acc","text_acc","raw_acc","q_acc","p95_acc","evidence_acc","krrb_acc"]].T.plot(kind="bar",legend=False)
plt.title("v59 Family-Quotient KRRB Accuracy")
plt.ylabel("accuracy")
plt.xticks(rotation=30,ha="right")
plt.tight_layout()
plt.show()

by_band.set_index("band")[["base_correct","raw_correct","q_correct","p95_correct","krrb_correct"]].plot(kind="bar", figsize=(14,5))
plt.title("v59 Accuracy by Band")
plt.ylabel("accuracy")
plt.xticks(rotation=30,ha="right")
plt.tight_layout()
plt.show()

pd.DataFrame({
    "raw":[results_df.raw_helped.sum(),results_df.raw_hurt.sum()],
    "q":[results_df.q_helped.sum(),results_df.q_hurt.sum()],
    "p95":[results_df.p95_helped.sum(),results_df.p95_hurt.sum()],
    "krrb":[results_df.krrb_helped.sum(),results_df.krrb_hurt.sum()],
},index=["helped","hurt"]).plot(kind="bar")
plt.title("v59 Help vs Hurt")
plt.ylabel("count")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

plt.figure(figsize=(10,4))
plt.hist(results_df["q_margin"], bins=20)
plt.title("Family-Quotient Residual Margin")
plt.xlabel("margin")
plt.ylabel("count")
plt.tight_layout()
plt.show()

heat=sweep_df.pivot_table(index="support_min", columns="alpha", values="krrb_acc", aggfunc="max")
display(heat)

plt.figure(figsize=(10,4))
plt.imshow(heat.values, aspect="auto")
plt.xticks(range(len(heat.columns)), heat.columns)
plt.yticks(range(len(heat.index)), heat.index)
plt.xlabel("alpha")
plt.ylabel("support_min")
plt.title("Best KRRB Accuracy by Alpha / Support Gate")
plt.colorbar()
plt.tight_layout()
plt.show()


In [ ]:
for row_id in interesting["id"].head(30):
    print("="*100)
    print("CASE:", row_id)
    display(results_df[results_df.id==row_id][[
        "id","base_id","band","gold_choice","choice_source_ids",
        "base_choice","base_source_id","base_correct","base_margin",
        "text_choice","text_source_id","text_correct","text_margin",
        "raw_choice","raw_source_id","raw_correct","raw_margin",
        "q_choice","q_source_id","q_correct","q_margin",
        "p95_choice","p95_source_id","p95_correct","p95_margin",
        "evidence_choice","evidence_source_id","evidence_correct","evidence_margin",
        "krrb_choice","krrb_source_id","krrb_correct","krrb_reason","omega",
        "support","gold_exact_phrase_leak","gold_slot_overlap","max_choice_slot_overlap"
    ]])
    print("NEED SLOT:")
    print(results_df[results_df.id==row_id]["need_slot_text"].iloc[0])
    display(branch_tables[row_id].sort_values("q_score", ascending=False))


## Readout

v59 separates three states:

$$
\Psi:
\text{family-quotient residual locks with support}
$$

$$
\Omega_1:
\text{raw residual locks but support is too weak}
$$

$$
\Omega_2:
\text{family neighbor collision remains unresolved}
$$

The critical publishable condition is:

$$
\boxed{
\text{krrb hurt}=0
}
$$

If this holds with gain over base, we now have a **safe outward-slot controller**.

If accuracy drops but hurt goes to zero, that is still progress:

$$
\text{safety before force}
$$

The next fold after v59 is not another benchmark. It is a slot compiler:

$$
(\text{prompt}) \rightarrow (\text{need slot})
$$

trained only on rows where:

$$
Q_{\text{real}} > Q_{\text{wrong}}
$$

and where KRRB has no hurt.
